# 13 · Hyperparameter tuning and final model selection

Tunes the poverty model (LightGBM, XGBoost; primary metric **PR-AUC**) and the VUP coverage-gap model (logistic regression, LightGBM, XGBoost; primary metric **ROC-AUC**). It then compares default, tuned and fairness-mitigated versions, and selects one final pipeline per model.

| | |
|---|---|
| **Data** | Development set only, the same 5 cluster-based folds. **The holdout is not used**: "holdout performance" in the selection is estimated out-of-fold, and the holdout is scored once in notebook 14. |
| **Search** | Optuna TPE sampler (seed 2026). Each study is capped by a trial count and a time limit, and every trial is logged with its fold scores (`reports/tuning_trials.csv`). |
| **Mitigation carried from notebook 12** | Group-aware recalibration (cross-fitted Platt scaling with disability, head-age, province and area intercepts) for the poverty model. Overall Platt recalibration for the VUP model, whose probabilities were too spread out. |
| **Outputs** | `models/poverty_vulnerability_pipeline_final.joblib`, `models/vup_gap_pipeline_final.joblib` (each with `.json`), `reports/tuning_results.csv`, `reports/tuning_trials.csv`, `data/processed/poverty_scores_final.parquet`, `reports/figures/tuning/*.png` |

**Final selection rule** (`src/models/tuning.py`). Every gate means "within one standard error", as the performance gate does.
1. **Performance:** mean fold primary metric within one fold standard error of the best.
2. **Fairness:**
   - recall gap no more than one bootstrap SE above the best eligible candidate;
   - worst-group recall no more than one bootstrap SE below it.
3. **Calibration:**
   - mean absolute group calibration gap within one SE of the best remaining candidate;
   - |slope − 1| within max(one SE, 0.05) of it.
4. **Simplicity:** the simplest survivor wins.

The SEs are cluster-bootstrap standard errors (100 draws) of each index, computed for the default model of each target.

Complexity scores:

| Model | Default | Tuned | Recalibration |
|---|---:|---:|---:|
| Logistic regression | 1 | +1 | +0.5 |
| LightGBM | 2 | +1 | +0.5 |
| XGBoost | 2.5 | +1 | +0.5 |

**How the rule was revised.**
- The rule was written before the full tuning run.
- A 3-trial smoke test showed that fixed tolerances (0.02 on the recall indices) were smaller than the sampling noise of those indices: one candidate was excluded by 0.0008.
- So the tolerances were changed to one bootstrap standard error before the full run (see `docs/tuning_decisions.md`).

**Caution.** The best tuned score is optimistic: it is the maximum over many trials evaluated on the same folds. The one-standard-error rule and the final holdout in notebook 14 guard against this.

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "common.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.models import common as cm
from src.models import fairness as fr
from src.models import plots as pl
from src.models import poverty_model as pm
from src.models import training as tr
from src.models import tuning as tu
from src.models import vup_gap_model as vm

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.4f}".format)
FIG = ROOT / "reports" / "figures" / "tuning"
FIG.mkdir(parents=True, exist_ok=True)
BUDGET = {"poverty": {"lightgbm": (60, 600), "xgboost": (60, 600)},
          "vup": {"logistic_regression": (60, 180), "lightgbm": (60, 300), "xgboost": (60, 300)}}  # (max trials, seconds)

df = cm.load_frame().merge(cm.load_splits(), on=["hhid", "clust"])
dev = df[df["split"].eq("development")].reset_index(drop=True)
hold = df[df["split"].eq("holdout")].reset_index(drop=True)
spec = pm.FEATURE_SETS["full"]
cm.assert_no_leakage(spec, "poverty")
folds = dev["fold"]
logs, studies = [], {}
print(f"Development {len(dev):,} households; budgets (trials, seconds): {BUDGET}")

## 1. Poverty model

In [ ]:
t0 = time.time()
pov_scores, pov_params = {}, {}
for name in ["logistic_regression", "lightgbm", "xgboost"]:
    pov_scores[(name, False)] = fr.cv_predict(tu.pipeline(spec, name), dev, spec.columns, "poor", folds)
    pov_params[(name, False)] = {}
print(f"Default models: {time.time() - t0:.0f}s")
for name, (n_trials, timeout) in BUDGET["poverty"].items():
    t0 = time.time()
    study, log = tu.run_study(name, spec, dev, "poor", folds, "pr_auc", n_trials, timeout)
    studies[("poverty", name)] = study
    logs.append(log.assign(model_target="poverty"))
    pov_params[(name, True)] = study.best_params
    pov_scores[(name, True)] = fr.cv_predict(tu.pipeline(spec, name, study.best_params), dev, spec.columns, "poor", folds)
    print(f"{name}: {len(study.trials)} trials in {time.time() - t0:.0f}s; best mean fold PR-AUC {study.best_value:.4f}; params {study.best_params}")

In [ ]:
rows = []
for (name, tuned), s in [(k, v) for k, v in pov_scores.items() if len(k) == 2]:
    base_c = tu.BASE_COMPLEXITY[name] + (tu.TUNED_EXTRA if tuned else 0)
    label = f"{name}{'_tuned' if tuned else '_default'}"
    rows.append(tu.candidate_row(label, s, dev, "poor", folds, "pr_auc", base_c, name, tuned, False, pov_params[(name, tuned)]))
    rec = fr.cross_fit_calibration(s, dev, "poor", folds)
    pov_scores[(name, tuned, "recal")] = rec
    rows.append(tu.candidate_row(label + "+group_recal", rec, dev, "poor", folds, "pr_auc", base_c + tu.RECAL_EXTRA, name, tuned, True, pov_params[(name, tuned)]))
pov_cand = pd.DataFrame(rows)
pov_se = fr.index_standard_errors(fr.add_groups(dev).assign(score=pov_scores[("lightgbm", False)].to_numpy()), "score", "poor", B=100)
pov_tol = tu.tolerances(pov_se)
print("Gate tolerances (one bootstrap SE, default LightGBM):", {k: round(v, 4) for k, v in pov_tol.items()})
pov_best, pov_table, pov_notes = tu.select_final(pov_cand, "pr_auc", pov_tol)
cols_show = ["candidate", "complexity", "pr_auc_mean", "pr_auc_se", "roc_auc", "recall_people_top20", "precision_top20", "calibration_slope",
             "calibration_in_the_large", "recall_gap", "worst_group_recall", "mean_abs_calibration_gap", "max_abs_calibration_gap",
             "gate_performance", "gate_fairness", "gate_calibration", "selected"]
display(pov_table.sort_values("pr_auc_mean", ascending=False)[cols_show])
print(f"Selected poverty model: {pov_best}" + (f"  (notes: {pov_notes})" if pov_notes else ""))

In [ ]:
# Fit the final poverty pipeline on the full development set
sel = pov_table.set_index("candidate").loc[pov_best]
base = cm.fit_pipeline(tu.pipeline(spec, sel["model"], sel["params"] or None), dev[spec.columns], dev["poor"], dev["weight"])
key = (sel["model"], bool(sel["tuned"]))
if sel["recalibrated"]:
    calibrator = fr.GroupCalibrator().fit(pov_scores[key], fr.add_groups(dev), dev["poor"], dev["weight"])
    final_pov = fr.GroupRecalibratedClassifier(base, calibrator)
    dev_score = pov_scores[key + ("recal",)]
else:
    final_pov = base
    dev_score = pov_scores[key]
meta = {"model": pov_best, "base_estimator": sel["model"], "tuned": bool(sel["tuned"]), "hyperparameters": sel["params"] or "library defaults",
        "group_recalibration": bool(sel["recalibrated"]), "calibration_groups": list(fr.CALIBRATION_GROUPS) if sel["recalibrated"] else [],
        "target": "poor", "feature_columns": spec.columns, "primary_metric": "pr_auc",
        "trained_on": f"development split: {len(dev):,} households (data/processed/model_splits.parquet)", "sample_weight": "household weight normalised to mean 1",
        "development_out_of_fold": {k: round(float(sel[k]), 4) for k in ["pr_auc_mean", "pr_auc_se", "roc_auc", "recall_people_top20", "calibration_slope", "recall_gap", "mean_abs_calibration_gap"]},
        "selection_rule": "tuning.select_final: 1-SE performance gate, fairness gate, calibration gate, simplest survivor",
        "usage": "joblib.load(path).predict_proba(frame[feature_columns])[:, 1]"}
tr.save_pipeline(final_pov, cm.MODELS_DIR / "poverty_vulnerability_pipeline_final.joblib", meta)
scores_final = pd.concat([dev[["hhid", "split", "fold"]].assign(poverty_score=dev_score.to_numpy()),
                          hold[["hhid", "split", "fold"]].assign(poverty_score=final_pov.predict_proba(hold[spec.columns])[:, 1])], ignore_index=True)
SCORES_FINAL = ROOT / "data" / "processed" / "poverty_scores_final.parquet"
scores_final.to_parquet(SCORES_FINAL, index=False)
print(f"Saved models/poverty_vulnerability_pipeline_final.joblib ({pov_best}) and {SCORES_FINAL.relative_to(ROOT)} "
      "(development: out-of-fold; holdout: features only, no labels used)")

## 2. VUP coverage-gap model

The poverty-vulnerability feature now comes from the final poverty model: out-of-fold for development households.

In [ ]:
hn = vm.load_population(scores_path=SCORES_FINAL)
hdev = hn[hn["split"].eq("development")].reset_index(drop=True)
vspec = vm.FEATURES
cm.assert_no_leakage(vspec, "vup")
vfolds = hdev["fold"]
t0 = time.time()
vup_scores, vup_params = {}, {}
for name in ["logistic_regression", "lightgbm", "xgboost"]:
    vup_scores[(name, False)] = fr.cv_predict(tu.pipeline(vspec, name), hdev, vspec.columns, "vup_any", vfolds)
    vup_params[(name, False)] = {}
print(f"Default models: {time.time() - t0:.0f}s")
for name, (n_trials, timeout) in BUDGET["vup"].items():
    t0 = time.time()
    study, log = tu.run_study(name, vspec, hdev, "vup_any", vfolds, "roc_auc", n_trials, timeout)
    studies[("vup", name)] = study
    logs.append(log.assign(model_target="vup_gap"))
    vup_params[(name, True)] = study.best_params
    vup_scores[(name, True)] = fr.cv_predict(tu.pipeline(vspec, name, study.best_params), hdev, vspec.columns, "vup_any", vfolds)
    print(f"{name}: {len(study.trials)} trials in {time.time() - t0:.0f}s; best mean fold ROC-AUC {study.best_value:.4f}; params {study.best_params}")

In [ ]:
rows = []
for (name, tuned), s in [(k, v) for k, v in vup_scores.items() if len(k) == 2]:
    base_c = tu.BASE_COMPLEXITY[name] + (tu.TUNED_EXTRA if tuned else 0)
    label = f"{name}{'_tuned' if tuned else '_default'}"
    rows.append(tu.candidate_row(label, s, hdev, "vup_any", vfolds, "roc_auc", base_c, name, tuned, False, vup_params[(name, tuned)]))
    rec = fr.cross_fit_calibration(s, hdev, "vup_any", vfolds, groups=())
    vup_scores[(name, tuned, "recal")] = rec
    rows.append(tu.candidate_row(label + "+platt", rec, hdev, "vup_any", vfolds, "roc_auc", base_c + tu.RECAL_EXTRA, name, tuned, True, vup_params[(name, tuned)]))
vup_cand = pd.DataFrame(rows)
vup_se = fr.index_standard_errors(fr.add_groups(hdev).assign(score=vup_scores[("logistic_regression", False)].to_numpy()), "score", "vup_any", B=100)
vup_tol = tu.tolerances(vup_se)
print("Gate tolerances (one bootstrap SE, default logistic regression):", {k: round(v, 4) for k, v in vup_tol.items()})
vup_best, vup_table, vup_notes = tu.select_final(vup_cand, "roc_auc", vup_tol)
vcols_show = ["candidate", "complexity", "roc_auc_mean", "roc_auc_se", "pr_auc", "recall_top20", "precision_top20", "calibration_slope",
              "calibration_in_the_large", "recall_gap", "worst_group_recall", "mean_abs_calibration_gap", "gate_performance", "gate_fairness",
              "gate_calibration", "selected"]
display(vup_table.sort_values("roc_auc_mean", ascending=False)[vcols_show])
print(f"Selected VUP model: {vup_best}" + (f"  (notes: {vup_notes})" if vup_notes else ""))

In [ ]:
vsel = vup_table.set_index("candidate").loc[vup_best]
vbase = cm.fit_pipeline(tu.pipeline(vspec, vsel["model"], vsel["params"] or None), hdev[vspec.columns], hdev["vup_any"], hdev["weight"])
vkey = (vsel["model"], bool(vsel["tuned"]))
if vsel["recalibrated"]:
    vcal = fr.GroupCalibrator(groups=()).fit(vup_scores[vkey], fr.add_groups(hdev), hdev["vup_any"], hdev["weight"])
    final_vup = fr.GroupRecalibratedClassifier(vbase, vcal)
else:
    final_vup = vbase
vmeta = {"model": vup_best, "base_estimator": vsel["model"], "tuned": bool(vsel["tuned"]), "hyperparameters": vsel["params"] or "library defaults",
         "platt_recalibration": bool(vsel["recalibrated"]), "population": "high-need households: distress_quintile_excl_vup == 5", "target": "vup_any",
         "feature_columns": vspec.columns, "poverty_score": "from models/poverty_vulnerability_pipeline_final.joblib (out-of-fold for training households)",
         "primary_metric": "roc_auc", "trained_on": f"development high-need households: {len(hdev):,} ({int(hdev['vup_any'].sum())} VUP recipients)",
         "development_out_of_fold": {k: round(float(vsel[k]), 4) for k in ["roc_auc_mean", "roc_auc_se", "pr_auc", "recall_top20", "calibration_slope", "mean_abs_calibration_gap"]},
         "interpretation": "low predicted probability = high-need profile relatively underserved by current VUP coverage; descriptive, not causal"}
tr.save_pipeline(final_vup, cm.MODELS_DIR / "vup_gap_pipeline_final.joblib", vmeta)
print(f"Saved models/vup_gap_pipeline_final.joblib ({vup_best})")

## 3. Logs and charts

In [ ]:
trials = pd.concat(logs, ignore_index=True)
trials.to_csv(ROOT / "reports" / "tuning_trials.csv", index=False)
results = pd.concat([pov_table.assign(model_target="poverty", primary_metric="pr_auc"),
                     vup_table.assign(model_target="vup_gap", primary_metric="roc_auc")], ignore_index=True)
results["params"] = results["params"].apply(json.dumps)
results.to_csv(ROOT / "reports" / "tuning_results.csv", index=False)
(ROOT / "reports" / "tuning_gate_tolerances.json").write_text(json.dumps({"poverty": pov_tol, "vup_gap": vup_tol,
                                                                          "poverty_index_se": pov_se, "vup_gap_index_se": vup_se}, indent=2))
print(f"reports/tuning_trials.csv: {len(trials)} trials; reports/tuning_results.csv: {len(results)} candidates")
display(trials.groupby(["model_target", "study"]).agg(trials=("number", "size"), best=("value", "max"), median=("value", "median"),
                                                      seconds=("user_attrs_seconds", "sum")))

In [ ]:
fig, axes = pl.figure("Tuning progress: best mean fold score so far", "Each dot is one trial (cluster-based 5-fold CV, development set); line = best so far; dashed = default model",
                      width=15, height=4.4, ncols=5)
defaults = {("poverty", "lightgbm"): pov_table.set_index("candidate").loc["lightgbm_default", "pr_auc_mean"],
            ("poverty", "xgboost"): pov_table.set_index("candidate").loc["xgboost_default", "pr_auc_mean"],
            ("vup", "logistic_regression"): vup_table.set_index("candidate").loc["logistic_regression_default", "roc_auc_mean"],
            ("vup", "lightgbm"): vup_table.set_index("candidate").loc["lightgbm_default", "roc_auc_mean"],
            ("vup", "xgboost"): vup_table.set_index("candidate").loc["xgboost_default", "roc_auc_mean"]}
for ax, (key, study) in zip(axes, studies.items()):
    v = np.array([t.value for t in study.trials if t.value is not None])
    ax.plot(range(len(v)), v, "o", color=pl.MODEL_COLORS[key[1]], alpha=0.45, markersize=4)
    ax.plot(range(len(v)), np.maximum.accumulate(v), "-", color=pl.INK["primary"], linewidth=2)
    ax.axhline(defaults[key], color=pl.INK["muted"], linestyle="--", linewidth=1)
    ax.set_title(f"{'Poverty' if key[0] == 'poverty' else 'VUP gap'}: {pl.MODEL_LABELS[key[1]]} ({'PR-AUC' if key[0] == 'poverty' else 'ROC-AUC'})",
                 fontsize=9, color=pl.INK["secondary"], loc="left")
    ax.set_xlabel("Trial", fontsize=8.5, color=pl.INK["secondary"])
    ax.grid(axis="both", color=pl.GRID)
path = pl.save(fig, FIG / "tuning_progress.png", top=0.74, bottom=0.16, left=0.05, wspace=0.3)
display(Image(str(path)))

In [ ]:
def candidate_chart(table, metric, title, path):
    t = table.sort_values(f"{metric}_mean")
    n = len(t)
    h = 1.9 + 0.32 * n
    fig, (ax,) = pl.figure(title, "Mean fold score with one standard error. Orange = selected; filled = passed all gates; hollow = failed a gate", width=10, height=h)
    y = np.arange(n)
    ax.hlines(y, t[f"{metric}_mean"] - t[f"{metric}_se"], t[f"{metric}_mean"] + t[f"{metric}_se"], color=pl.BASELINE, linewidth=2)
    for yi, (_, r) in zip(y, t.iterrows()):
        color = pl.SERIES[1] if r["selected"] else pl.SERIES[0]
        ax.plot(r[f"{metric}_mean"], yi, "o", markersize=8, color=color, markerfacecolor=color if r["survivor"] else pl.SURFACE, markeredgewidth=2)
        ax.text(r[f"{metric}_mean"] + r[f"{metric}_se"] + 0.001, yi, f" {r[f'{metric}_mean']:.3f}", va="center", fontsize=8)
    ax.set_yticks(y, t["candidate"], fontsize=8.5)
    top = t[f"{metric}_mean"].idxmax()
    ax.axvline(t.loc[top, f"{metric}_mean"] - t.loc[top, f"{metric}_se"], color=pl.INK["muted"], linestyle="--", linewidth=1)
    return pl.save(fig, path, top=1 - 1.25 / h, bottom=0.5 / h, left=0.32)

display(Image(str(candidate_chart(pov_table, "pr_auc", "Poverty model candidates (development, out-of-fold PR-AUC)", FIG / "poverty_candidates.png"))))
display(Image(str(candidate_chart(vup_table, "roc_auc", "VUP coverage-gap candidates (development, out-of-fold ROC-AUC)", FIG / "vup_candidates.png"))))